# Inverted indexes & Boolean retrieval — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Postings lists turn Boolean search into set arithmetic
A tiny corpus makes the index visible: build postings, run AND/OR/NOT, watch sorted-list intersection, count document frequency, and compare indexed work with naive scanning.

In [ ]:
# 1) Build an inverted index from four toy documents
terms=['neural','search','graph','retrieval']
docs=[{'neural','search'},{'search','graph'},{'neural','retrieval'},{'graph','retrieval','search'}]
post={t:[i+1 for i,d in enumerate(docs) if t in d] for t in terms}
plt.figure(figsize=(6,3)); plt.bar(terms,[len(post[t]) for t in terms]); plt.ylabel('posting length'); plt.title('Document frequency from postings')
assert post['search']==[1,2,4] and post['retrieval']==[3,4]

In [ ]:
# 2) Boolean AND is set intersection
ans=sorted(set(post['search']) & set(post['graph']))
plt.figure(figsize=(6,3)); plt.bar(['search','graph','AND'],[len(post['search']),len(post['graph']),len(ans)],color=['tab:blue','tab:orange','tab:green']); plt.title('search AND graph -> docs 2 and 4')
assert ans==[2,4]

In [ ]:
# 3) Boolean OR expands with set union
ans=sorted(set(post['neural']) | set(post['retrieval']))
plt.figure(figsize=(6,3)); plt.bar(['neural','retrieval','OR'],[len(post['neural']),len(post['retrieval']),len(ans)]); plt.title('neural OR retrieval -> docs 1, 3, 4')
assert ans==[1,3,4]

In [ ]:
# 4) Boolean NOT is complement against the corpus ids
universe=set(range(1,5)); ans=sorted(universe-set(post['search']))
plt.figure(figsize=(6,3)); plt.bar(['all docs','search docs','NOT search'],[4,len(post['search']),len(ans)],color=['gray','tab:red','tab:green']); plt.title('NOT search leaves only doc 3')
assert ans==[3]

In [ ]:
# 5) Sorted postings make intersection a two-pointer walk
A=post['search']; B=post['graph']; i=j=steps=0; out=[]
while i<len(A) and j<len(B):
    steps+=1
    if A[i]==B[j]: out.append(A[i]); i+=1; j+=1
    elif A[i]<B[j]: i+=1
    else: j+=1
plt.figure(figsize=(6,3)); plt.bar(['two-pointer steps','nested comparisons'],[steps,len(A)*len(B)]); plt.title('Sorted postings reduce comparisons')
assert out==[2,4] and steps==3 and len(A)*len(B)==6